# NDI-F at the million-row scale (Google Colab Pro)

Runs the experiments of `large/ndi_large.py` on seven large public datasets (130,000 to 11,000,000 rows):
accuracy and time against the number of rows, the GPU imputers on 50,000-row samples, the streaming (online) scenario and a
downstream task. Everything is written to **Google Drive** and is resumable: if the session ends, run the cells again and the
run continues where it stopped.

**Runtime:** `Runtime → Change runtime type → A100 (or L4) GPU` and **High-RAM**. The CPU experiments (ACC, ONLINE, DOWN) need
the RAM (the largest tables are 11,000,000 × 28); the GPU is used by the DEEP step only, so ACC/ONLINE can also run on a CPU
High-RAM runtime first and DEEP on a GPU runtime afterwards.

**Disk (Drive):** about 4 GB of downloads (deleted after conversion) and 3 GB of cached `.npy` tables; results are a few MB.

**Time (rough, A100 High-RAM):** data download and conversion 30–60 min (once); ACC on all seven datasets 6–10 h (EM-FA and
EM-PPCA to convergence are limited to 2,000,000 rows and GaussCM to 1,000,000 by default; raise `MAX_N_*` to go further);
ONLINE 1–2 h; DEEP about 1 h per dataset with the budgets below; DOWN about 30 min per dataset.

**Second session (7 November 2026): upload this notebook and choose Runtime → Run all.** You are asked twice at the start:
the Google Drive permission (cell 1) and the code zip (cell 2, choose `ndi-imputation-v4.1-github.zip`). After that it runs on its
own for about 3.5 hours. Cells 4, 5, 6 and 8 finished in the first session and are skipped automatically (switches in cell 1).
If the session disconnects, choose Run all again: only the Drive permission is asked again and everything continues where it stopped.
At the end, download `openml_datasets.zip` and `results_large.zip` from the `ndi_large` folder of your Drive and send them to Claude.

In [ ]:
# 1) Google Drive and folders
import os
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/ndi_large'          # everything of this notebook lives here
for d in ['data', 'results']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
os.environ['DATA_DIR'] = f'{BASE}/data'
os.environ['OUT_DIR']  = f'{BASE}/results'
os.chdir(BASE)

# Switches for "Run all".  The first session finished the data download (cell 4), the smoke test (5), ACC + ONLINE (6) and
# DOWN (8); their results are on Drive, so these cells are skipped.  Set SKIP_DONE = False to run everything from scratch
# (they would only find the finished rows and skip them, in a few minutes).
SKIP_DONE = True
RUN_SINKHORN = False      # cell 7c, optional: about 8 hours on a T4

In [ ]:
# 2) The code (ndi-imputation, newest version). If the copy on Drive is missing or older than this notebook, it is replaced:
#    from GitHub once version 4.1 is pushed there, otherwise from the zip you choose with the button that appears below
#    (ndi-imputation-v4.1-github.zip, the newest one). The results on Drive (ndi_large/results) are never touched.
import os, shutil, zipfile
VERSION = '2026-11-07b'                               # LARGE_VERSION of large/ndi_large.py this notebook expects
def current():
    p = 'ndi-imputation/large/ndi_large.py'
    return os.path.exists(p) and VERSION in open(p).read()
def install_zip(path):
    tmp = '/content/_code'; shutil.rmtree(tmp, ignore_errors=True); zipfile.ZipFile(path).extractall(tmp)
    src = f'{tmp}/ndi-imputation'
    for root, dirs, fns in os.walk(src):
        rel = os.path.relpath(root, src); os.makedirs(os.path.join('ndi-imputation', rel), exist_ok=True)
        for f in fns: shutil.copyfile(os.path.join(root, f), os.path.join('ndi-imputation', rel, f))
    print('code installed from', os.path.basename(path))
if not current():
    if not os.path.exists('ndi-imputation'):
        !git clone -q https://github.com/kurbankotan/ndi-imputation.git
    elif os.path.exists('ndi-imputation/.git'):
        !cd ndi-imputation && git pull -q 2>/dev/null || true
if not current():
    print('The code on Drive is older than this notebook: choose ndi-imputation-v4.1-github.zip (the newest) from your computer')
    from google.colab import files
    shutil.rmtree('/content/up', ignore_errors=True); os.makedirs('/content/up'); _cwd = os.getcwd(); os.chdir('/content/up')
    _up = files.upload(); os.chdir(_cwd)
    install_zip('/content/up/' + [n for n in _up if n.endswith('.zip')][0])
assert current(), 'still an older version of the code: upload the newest ndi-imputation-v4.1-github.zip'
%cd ndi-imputation


In [ ]:
# 3) Packages (about 3-5 minutes, once per session); the ReMasker code is needed for the DEEP step only
!pip -q install "pandas<3" "scikit-learn>=1.5,<1.8" "hyperimpute==0.1.17" timm
if not os.path.exists('remasker/remasker_impute.py'):
    !git clone -q https://github.com/tydusky/remasker.git
!sed -i 's/^from tkinter import E$/# from tkinter import E/' remasker/model_mae.py
!sed -i 's/np\.float\b/float/g; s/np\.int\b/int/g; s/np\.bool\b/bool/g' remasker/remasker_impute.py remasker/model_mae.py remasker/utils.py remasker/plugin_mae.py
import multiprocessing, torch, psutil
print("CPU cores:", multiprocessing.cpu_count(), "| RAM GB:", round(psutil.virtual_memory().total / 1e9), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# 3b) One-time, for the ADAC paper (not for this notebook): export the 15 OpenML tables of the main benchmark (1-2 minutes).
#     Writes ndi_large/openml_datasets.zip on Drive; download it and send it to Claude.
try:
    import os, zipfile
    from sklearn.datasets import fetch_openml
    IDS = {'iris': 61, 'wine': 187, 'glass': 41, 'ecoli': 39, 'ionosphere': 59, 'wdbc': 1510, 'diabetes': 37, 'vehicle': 54, 'yeast': 181,
           'banknote': 1462, 'qsar-biodeg': 1494, 'wine-quality-red': 40691, 'segment': 36, 'spambase': 44, 'MagicTelescope': 1120}
    os.makedirs('/content/openml_csv', exist_ok=True)
    for name, i in IDS.items():
        d = fetch_openml(data_id=i, as_frame=True, parser='auto')            # exactly as ndi_benchmark.load_openml
        df = d.data.select_dtypes(include='number').copy(); df['__target__'] = d.target
        df.to_csv(f'/content/openml_csv/{name}.csv', index=False); print(f'{name:<18s} {df.shape}')
    out = '/content/drive/MyDrive/ndi_large/openml_datasets.zip'
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
        for name in IDS: z.write(f'/content/openml_csv/{name}.csv', f'{name}.csv')
    print('ready:', out)
except Exception as e:                                    # a failure here must not stop Run all
    print('!! OpenML export failed:', type(e).__name__, e, '- the other cells continue')

In [ ]:
# 4) Datasets -> Drive (once; 30-60 minutes). Order: small to large. Re-running skips the cached ones.
if SKIP_DONE:
    print('cell 4 skipped: the datasets are already on Drive')
else:
    !python large/get_large_data.py MiniBooNE YearPredictionMSD covertype KDDCup99 SUSY HEPMASS HIGGS 2>&1 | grep -v -i "warning" | tail -n 30

In [ ]:
# 5) Smoke test on synthetic data (about 2 minutes) -> results_large_quick/ inside the repository folder (not the Drive results)
if SKIP_DONE:
    print('cell 5 skipped: smoke test done in the first session')
else:
    !QUICK=1 OUT_DIR=results_large_quick EXP=ACC,ONLINE,DOWN python large/ndi_large.py 2>&1 | grep -v -i "warning\|deprecat" | tail -n 15

In [ ]:
# 6) ACC + ONLINE on all seven datasets (CPU, High-RAM; many hours; resumable). Start with the small ones:
#    DATASETS=MiniBooNE,YearPredictionMSD,covertype first, then KDDCup99,SUSY,HEPMASS,HIGGS.
#    MAX_N_EM_FA=11000000 MAX_N_EM_PPCA=11000000 MAX_N_GaussCM=5000000 would run those methods on the full tables (slow).
#    Output streams line by line (python -u, grep --line-buffered); the same lines go to results/log_acc_online.txt on Drive.
if SKIP_DONE:
    print('cell 6 skipped: ACC and ONLINE finished in the first session')
else:
    !EXP=ACC,ONLINE DATASETS=MiniBooNE,YearPredictionMSD,covertype,KDDCup99,SUSY,HEPMASS,HIGGS python -u large/ndi_large.py 2>&1 | grep --line-buffered -v -i "warning\|deprecat" | tee -a $OUT_DIR/log_acc_online.txt

In [ ]:
# 7) DEEP, second pass: MIWAE (the blocks that ran out of GPU memory) and GAIN with a budget scaled to 50,000 rows.
#    Each GPU method now runs in its own process, so the memory left behind by the previous method cannot cause the
#    out-of-memory errors of the first pass. Rows already done (ReMasker@100, HyperImpute, MIWAE@200 that worked) are skipped;
#    rows that ended in an error are recomputed. About 2.5 hours on a T4, less on an L4 or A100.
!EXP=DEEP DEEP_N=50000 DEEP_METHODS=MIWAE,GAIN MIWAE_EPOCHS=200 GAIN_ITERS=50000 DATASETS=MiniBooNE,YearPredictionMSD,covertype,KDDCup99,SUSY,HEPMASS,HIGGS python -u large/ndi_large.py 2>&1 | grep --line-buffered -v -i "warning\|deprecat" | tee -a $OUT_DIR/log_deep.txt

In [ ]:
# 7c) OPTIONAL: Sinkhorn imputation with 2,000 updates (about 40 visits per row; the first pass had 200, i.e. 4 visits per row,
#     and stayed at the mean level). About 8 hours on a T4. Can be skipped: the comparison stands without it.
if not RUN_SINKHORN:
    print('cell 7c skipped: Sinkhorn is optional (RUN_SINKHORN = True in cell 1 runs it)')
else:
    !EXP=DEEP DEEP_N=50000 DEEP_METHODS=Sinkhorn SINKHORN_ITERS=2000 DATASETS=MiniBooNE,YearPredictionMSD,covertype,KDDCup99,SUSY,HEPMASS,HIGGS python -u large/ndi_large.py 2>&1 | grep --line-buffered -v -i "warning\|deprecat" | tee -a $OUT_DIR/log_deep.txt

In [ ]:
# 7d) GaussCM (full Gaussian conditional mean) on the full tables, 30 % MCAR: KDDCup99 4.9 M, SUSY 5 M, HEPMASS 10.5 M and
#     HIGGS 11 M rows. CPU only, High-RAM; about 40 minutes. Only the missing rows are computed.
!EXP=ACC MAX_N_GaussCM=11000000 DATASETS=KDDCup99,SUSY,HEPMASS,HIGGS python -u large/ndi_large.py 2>&1 | grep --line-buffered -v -i "warning\|deprecat" | tee -a $OUT_DIR/log_acc_online.txt

In [ ]:
# 7e) Progress (run in a SECOND notebook on the same Drive while a step is running, or afterwards)
import os, pandas as pd
R = '/content/drive/MyDrive/ndi_large/results'
for f in ['acc_long.csv', 'deep_long.csv', 'downstream_long.csv', 'online_long.csv']:
    p = f'{R}/{f}'
    if not os.path.exists(p): continue
    d = pd.read_csv(p); d = d[d.dataset != 'synthetic']; print(f'{f}: {len(d)} rows')
    if 'error' in d and d.error.notna().any(): print('  rows with errors:'); print(d[d.error.notna()][['dataset', 'method', 'error']].to_string())
    if f == 'deep_long.csv': print(d.groupby('method').agg(blocks=('rmse', 'size'), ok=('rmse', lambda x: x.notna().sum()), mean_s=('time_s', 'mean')).round(0))

In [ ]:
# 8) DOWN (optional): HistGradientBoosting on the 100,000-row samples, imputed by the cheap methods (about 5-8 hours)
if SKIP_DONE:
    print('cell 8 skipped: DOWN finished in the first session')
else:
    !EXP=DOWN DATASETS=MiniBooNE,YearPredictionMSD,covertype,KDDCup99,SUSY,HEPMASS,HIGGS python -u large/ndi_large.py 2>&1 | grep --line-buffered -v -i "warning\|deprecat" | tee -a $OUT_DIR/log_down.txt

In [ ]:
# 9) Summaries and figures (large_tables.json, summary_*.csv, figL1-3) in the results folder on Drive
!python large/make_large_tables.py $OUT_DIR $OUT_DIR
import pandas as pd
pd.set_option('display.width', 220)
print(pd.read_csv(f'{os.environ["OUT_DIR"]}/summary_rmse_wide.csv').round(3).to_string())
print(pd.read_csv(f'{os.environ["OUT_DIR"]}/summary_time.csv').round(1).to_string())
from IPython.display import Image, display
for f in ['figL1_error_vs_n.png', 'figL2_time_vs_n.png', 'figL3_online.png']:
    p = f'{os.environ["OUT_DIR"]}/{f}'
    if os.path.exists(p): display(Image(p, width=1100))

In [ ]:
# 10) Pack the results (Drive keeps them anyway)
!cd $OUT_DIR/.. && zip -q -r results_large.zip results -x "*.npy"
print('zipped to', f'{BASE}/results_large.zip')
print('\nFINISHED. From the ndi_large folder of your Google Drive, download openml_datasets.zip and results_large.zip and send them to Claude.')